# Practice Lab: Containerization with Docker (Lesson 12.5)

Module 12 . 4 exercises . Image sizes + layer caching + multi-stage + CI/CD


# Lesson 12.5: Containerization with Docker

4 exercises: 2E + 1M + 1C

All exercises simulate Docker operations (no daemon required).


In [ ]:
# No special imports needed - pure Python simulations


---
## Exercise 1 (Easy): Image Size Comparison


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class SizeAnalyzer:
    def __init__(self): self.imgs={}
    def build(self,name,base,deps,app,junk=0):
        self.imgs[name]={"base":base,"deps":deps,"app":app,"junk":junk,"total":base+deps+app+junk}

a=SizeAnalyzer()
a.build("naive (python:3.12)",1020,180,5,85)
a.build("production (slim)",150,120,3,0)
a.build("multi-stage (slim)",150,95,3,0)
a.build("alpine (broken numpy)",50,250,3,0)

print("Docker Image Size Comparison:")
print(f"  {'Image':<25} {'Base':>6} {'Deps':>6} {'App':>5} {'Junk':>5} {'Total':>7}")
for n,s in a.imgs.items():
    print(f"  {n:<25} {s['base']:>5}M {s['deps']:>5}M {s['app']:>4}M {s['junk']:>4}M {s['total']:>6}M")

naive=a.imgs["naive (python:3.12)"]["total"]; multi=a.imgs["multi-stage (slim)"]["total"]
print(f"\n  Reduction: {naive}M -> {multi}M ({(1-multi/naive)*100:.0f}%)")
print(f"  6 fixes: slim base, .dockerignore, --no-cache-dir, multi-stage, non-root, pin versions")


</details>


---
## Exercise 2 (Easy): Layer Caching Analysis


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class LayerSim:
    def __init__(self,name): self.name=name; self.layers=[]
    def add(self,instr,time_s,trigger=""): self.layers.append({"i":instr,"t":time_s,"tr":trigger})
    def build(self,changed):
        broken=False; total=0; results=[]
        for l in self.layers:
            if broken or (l["tr"] and l["tr"] in changed):
                broken=True; total+=l["t"]; results.append((l["i"],"REBUILD",l["t"]))
            else: results.append((l["i"],"CACHED",0))
        return results,total

naive=LayerSim("Naive")
naive.add("FROM python:3.12",0); naive.add("COPY . .",2,"any"); naive.add("pip install",45,"req"); naive.add("CMD",0)

prod=LayerSim("Production")
prod.add("FROM slim",0); prod.add("apt-get",8); prod.add("COPY req.txt",0.1,"req")
prod.add("pip install",40,"req"); prod.add("COPY . .",1,"any"); prod.add("CMD",0)

print("Layer Caching Analysis:")
for scenario,changed in [("Code change","any"),("Dep change","req"),("No change","")]:
    nr,nt=naive.build(changed); pr,pt=prod.build(changed)
    print(f"\n  {scenario}:")
    print(f"    Naive: {nt:.0f}s | Production: {pt:.1f}s",end="")
    if nt>0 and pt>0: print(f" | {nt/pt:.0f}x faster")
    elif pt==0: print(f" | fully cached!")
    else: print()


</details>


---
## Exercise 3 (Medium): Multi-Stage Build


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Multi-Stage Build (numpy+scipy):")
components=[("python:3.12-slim base",150,150),("build-essential",120,0),("dev headers",25,0),
    ("pip cache",45,0),("numpy compiled",65,65),("scipy compiled",85,85),
    ("fastapi+uvicorn",25,25),("google-genai",30,30),("app code",3,3)]

print(f"  {'Component':<25} {'Single':>8} {'Multi':>8}")
s_total=m_total=0
for name,single,multi in components:
    s_total+=single; m_total+=multi
    saved="*" if single!=multi else " "
    print(f"  {name:<25} {single:>7}M {multi:>7}M {saved}")
print(f"  {'TOTAL':<25} {s_total:>7}M {m_total:>7}M")
print(f"\n  Savings: {s_total-m_total}MB ({(1-m_total/s_total)*100:.0f}%)")
print(f"  No gcc in production = smaller attack surface")


</details>


---
## Exercise 4 (Challenge): CI/CD Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
stages=[("Checkout",5,0),("Lint (ruff)",15,0),("Tests (pytest)",30,0),
    ("Docker build",90,0.008),("Vuln scan (trivy)",20,0),("Push to GCR",25,0.002),
    ("Deploy canary 5%",30,0.010),("Health check",30,0),("Promote 100%",15,0.005)]

print("CI/CD Pipeline:")
print(f"  {'Stage':<22} {'Time':>6} {'Cost':>7}")
tt=tc=0
for name,t,c in stages:
    tt+=t; tc+=c; print(f"  {name:<22} {t//60}m{t%60:02d}s ${c:.3f}")
print(f"  {'TOTAL':<22} {tt//60}m{tt%60:02d}s ${tc:.3f}")
print(f"\n  GitHub Actions: $0.008/min private = ${tt/60*0.008:.3f}/run")
print(f"  Free: 2,000 min/month (public repos)")
print(f"  Rollback: gcloud run services update-traffic --to-revisions PREV=100")
print(f"  Trigger: error_rate > 2% OR latency_p99 > 5s")
print(f"  Rollback time: ~15s (traffic shift, no rebuild)")


</details>
